# Imports

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import keras
import numpy as np
import pandas as pd
import transformers
import tokenizers
# import tensorflow as tf
import torch
import tqdm.notebook as tqdm
import sklearn.model_selection
import matplotlib.pyplot as plt

from pathlib import Path
import gc, traceback, sys

In [2]:
keras.__version__, keras.backend.backend()

('3.15.1', 'torch')

In [3]:
keras.mixed_precision.set_global_policy('mixed_float16')

In [4]:
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         tf.config.experimental.set_memory_growth(gpus[0], True)
#         print(gpus)
#     except:
#         print("No GPU")

gpus = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
if gpus:
    try:
        torch.cuda.set_per_process_memory_fraction(0.8)
        for i in range(torch.cuda.device_count()):
           print(torch.cuda.get_device_properties(i))
    except:
        print("No GPU")


_CudaDeviceProperties(name='NVIDIA GeForce RTX 3060', major=8, minor=6, total_memory=12287MB, multi_processor_count=28, uuid=9c6396a2-0e5e-4445-8ae5-e68074710ca5, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=2MB)


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [5]:
class Tokenizer(transformers.GPT2Tokenizer):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.pad_token_id = self.eos_token_id
        self._tokenizer.post_processor = tokenizers.processors.TemplateProcessing(
            single=f"{self.bos_token} $A {self.eos_token}",
            pair=f"{self.bos_token} $A {self.bos_token} $B {self.eos_token}",
            special_tokens=[
                (self.bos_token, self.bos_token_id),
                (self.eos_token, self.eos_token_id),
            ],
        )
        


In [6]:
tokenizer = Tokenizer.from_pretrained('gpt2')

In [7]:
DATASET_PATH = Path("./dataset")

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

In [8]:
def get_total_sentences():
    en_sentences = pd.read_csv(DATASET_PATH / "eng_sentences.tsv", sep="\t", header=None, names=["en_id", "lang", "sentence"]).drop(columns=["lang"])
    ru_sentences = pd.read_csv(DATASET_PATH / "rus_sentences.tsv", sep="\t", header=None, names=["ru_id", "lang", "sentence"]).drop(columns=["lang"])
    links = pd.read_csv(DATASET_PATH / "links.tsv", sep="\t", header=None, names=["en_id", "ru_id"])
    return (
        links
        .merge(en_sentences, on="en_id", how="inner")
        .merge(ru_sentences, on="ru_id", how="inner")
        .drop(columns=["en_id", "ru_id"])
        .rename(columns={"sentence_x": "en", "sentence_y": "ru"})
    )

In [9]:
total_sentences = get_total_sentences()
total_sentences.head()

,en,ru
0,Let's try something.,Давайте что-нибудь попробуем!
1,Let's try something.,Давайте что-нибудь попробуем.
2,Let's try something.,Давай кое-что попробуем.
3,I have to go to sleep.,Мне пора идти спать.
4,I have to go to sleep.,Мне нужно идти спать.


In [10]:
len(total_sentences), len(total_sentences.groupby("en")), len(total_sentences.groupby("ru"))

(829668, 544615, 710900)

In [11]:
_i = -2
_en = total_sentences.iloc[_i].en
_ru = total_sentences.iloc[_i].ru
(
    _en, len(_en), len(tokenizer.encode(_en, add_special_tokens=False)), len(tokenizer.encode(_en)),
    _ru, len(_ru), len(tokenizer.encode(_ru, add_special_tokens=False)), len(tokenizer.encode(_ru))
)

("The old woman was evicted from an apartment she'd lived her whole life in.",
 74,
 17,
 19,
 'Пожилую женщину выселили из квартиры, в которой она прожила всю жизнь.',
 70,
 76,
 78)

# Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [10]:
class Dataset(keras.utils.PyDataset):

    def __init__(
            self,
            X: pd.Series,
            y: pd.Series,
            batch_size: int,
            seed: int | None = None, 
            shuffle: bool = False, 
            **kwargs
    ):
        super().__init__(**kwargs)
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.seed = seed
        self.rng = np.random.default_rng(seed)

        self.X = X
        self.y = y

        self.image_index = np.arange(len(self.X))

        self.on_epoch_end()

    def __len__(self):
        return (len(self.image_index) + self.batch_size - 1) // self.batch_size

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.image_index)
    
    def __getitem__(self, index: int):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch = self.image_index[start:end]

        _X = tokenizer.encode(self.X.iloc[batch].to_list())
        max_len_X = int(np.quantile(list(map(len, _X)), 0.9))
        
        _X = keras.utils.pad_sequences(_X, maxlen=max_len_X, padding='post', truncating='post', value=tokenizer.eos_token_id)
        
        _y = tokenizer.encode(self.y.iloc[batch].to_list())
        max_len_y = int(np.quantile(list(map(len, _y)), 0.9))
        
        _mask = keras.utils.pad_sequences(_y, maxlen=max_len_y, padding='post', truncating='post', value=-1)
        _mask = np.where(_mask == -1, 0, 1)
        
        _y = keras.utils.pad_sequences(_y, maxlen=max_len_y, padding='post', truncating='post', value=tokenizer.eos_token_id)

        _y_pred = _y[:, 1:]
        _y_pred = keras.utils.pad_sequences(_y_pred, maxlen=max_len_y, padding='post', truncating='post', value=tokenizer.eos_token_id)
        _y_pred[:, -1] = tokenizer.eos_token_id

        return (_X, _y), _y_pred, _mask


In [11]:
def get_datasets(df, batch_size: int=32, seed: int=None) -> tuple[Dataset, Dataset]:    
    gss = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(df, groups=df.en))

    _train = Dataset(df.ru.iloc[train_idx], df.en.iloc[train_idx], batch_size=batch_size, seed=seed, shuffle=True)
    _test = Dataset(df.ru.iloc[test_idx], df.en.iloc[test_idx], batch_size=batch_size, seed=seed, shuffle=False)

    return _train, _test

In [12]:
train, test = get_datasets(total_sentences, batch_size=512, seed=42)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [13]:
def get_name(prefix: str | None = None, suffix: str | None = None, separator: str = '_') -> str | None:
    return prefix and prefix + separator + suffix or suffix or None

In [20]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    attention: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> tuple[keras.Model, keras.Model, keras.Model] :
    '''Creates a model with RNN architecture for sequence to sequence classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The trining model, inference encoder, inference deocder'''

    def get_rnn(*, cell_units: int, name_suffix: str):
        cells = [cell_type(cell_units) for _ in range(n_stacks)]
        stacked_cells = keras.layers.StackedRNNCells(cells)
        return keras.layers.RNN(
            stacked_cells,
            return_sequences=True,
            return_state=True,
            name=get_name("rnn_cell", name_suffix),
        )

    # enc layers
    inputs_enc = x = keras.layers.Input((None,), dtype='int32', name='input_enc')
    x = keras.layers.Embedding(n_tokens, units, name='embedding_enc')(x)

    rnn_enc = get_rnn(
        cell_units=units,
        name_suffix="enc",
    )

    if bidirectional:
        rnn_enc = keras.layers.Bidirectional(rnn_enc, name="bidirectional_enc")

    enc_seq, *enc_states = rnn_enc(x)

    if bidirectional:
        states_per_dir = len(enc_states) // (2 * n_stacks)
        concatenated_states = []
        
        for i in range(n_stacks):
            for j in range(states_per_dir):
                fwd_idx = i * 2 * states_per_dir + j
                bwd_idx = fwd_idx + states_per_dir
                
                fwd_state = enc_states[fwd_idx]
                bwd_state = enc_states[bwd_idx]
                
                concat_state = keras.layers.Concatenate(name=f"concat_state_{i}_{j}")([fwd_state, bwd_state])
                concatenated_states.append(concat_state)
                
        enc_states = concatenated_states

    # dec layers
    inputs_dec = x = keras.layers.Input((None,), dtype='int32', name='input_dec')
    x = keras.layers.Embedding(n_labels, units, name='embedding_dec')(x)

    rnn_dec = get_rnn(
        cell_units=units * (2 if bidirectional else 1),
        name_suffix="dec",
    )

    dec_seq, *dec_states = rnn_dec(x, initial_state=enc_states)
    x = dec_seq

    if attention:
        ctx = keras.layers.AdditiveAttention(name="additive_attention")([dec_seq, enc_seq])
        x = keras.layers.Concatenate(name="concat_dec_attn")([dec_seq, ctx])
 
    x = keras.layers.Dropout(0.2, name="dropout")(x)
    pred = keras.layers.Dense(n_labels, name='output')(x)

    # train model
    training_model = keras.Model(
        inputs=[inputs_enc, inputs_dec], 
        outputs=pred, 
        name=name
    )

    # enc inference
    encoder_model = keras.Model(
        inputs=inputs_enc,
        outputs=[enc_seq] + enc_states,
        name=f"{name or 'inference'}_encoder"
    )

    # dec inference
    inf_dec_input = x = keras.layers.Input((None,), dtype='int32', name='inf_input_dec')
    
    inf_dec_states = [
        keras.layers.Input(shape=state.shape[1:], dtype=state.dtype, name=f'inf_dec_state_{i}')
        for i, state in enumerate(enc_states)
    ]
    
    x = training_model.get_layer('embedding_dec')(x)
    
    inf_dec_seq, *inf_new_dec_states = training_model.get_layer(get_name("rnn_cell", "dec"))(
        x,
        initial_state=inf_dec_states
    )
    
    x = inf_dec_seq
    if attention:
        inf_enc_seq_input = keras.layers.Input(
            shape=enc_seq.shape[1:], 
            dtype=enc_seq.dtype, 
            name='inf_enc_seq'
        )
        inf_ctx = training_model.get_layer("additive_attention")([inf_dec_seq, inf_enc_seq_input])
        x = training_model.get_layer("concat_dec_attn")([inf_dec_seq, inf_ctx])
        
        decoder_inputs_list = [inf_dec_input, inf_enc_seq_input] + inf_dec_states
    else:
        decoder_inputs_list = [inf_dec_input] + inf_dec_states

    x = training_model.get_layer("dropout")(x, training=False)
    inf_predictions = training_model.get_layer("output")(x)

    decoder_model = keras.Model(
        inputs=decoder_inputs_list,
        outputs=[inf_predictions] + inf_new_dec_states,
        name=f"{name or 'inference'}_decoder" if name else "decoder_model"
    )
    
    return training_model, encoder_model, decoder_model

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

In [21]:
models = [
    get_model(
        units=128,
        n_tokens=len(tokenizer.get_vocab()),
        n_labels=len(tokenizer.get_vocab()),
        n_stacks=1,
        bidirectional=False,
        attention=att,
        name=f"GRU-Att-{att}",
        cell_type=keras.layers.GRUCell
    )
    for att in [False, True]
]

In [26]:
models[1][0].summary()

Model: "GRU-Att-True"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_enc           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_dec           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_enc       │ (None, None, 128) │  6,432,896 │ input_enc[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_dec       │ (None, None, 128) │  6,432,896 │ input_dec[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_cell_enc (RNN)  │ [(None, None,     │     99,072 │ embedding_enc[0]… │
│                     │ 128), (None,      │            │                   │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_cell_dec (RNN)  │ [(None, None,     │     99,072 │ embedding_dec[0]… │
│                     │ 128), (None,      │            │ rnn_cell_enc[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention  │ (None, None, 128) │        128 │ rnn_cell_dec[0][… │
│ (AdditiveAttention) │                   │            │ rnn_cell_enc[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_dec_attn     │ (None, None, 256) │          0 │ rnn_cell_dec[0][… │
│ (Concatenate)       │                   │            │ additive_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None, 256) │          0 │ concat_dec_attn[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │ 12,916,049 │ dropout[0][0]     │
│                     │ 50257)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,980,113 (99.11 MB)

 Trainable params: 25,980,113 (99.11 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
for model in models:
    model[0].compile(
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=keras.optimizers.Adam(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(),
        ]
    )

In [28]:
SAVE_MODEL_PATH = Path("./saved")

In [29]:
with torch.no_grad():
    torch.cuda.empty_cache()
torch.cuda.empty_cache(), traceback.clear_frames(sys.last_traceback)
gc.collect()

6202

In [30]:
torch.cuda.memory_allocated(), torch.cuda.memory_reserved()

(182120448, 446693376)

In [ ]:
for _model in models:
    model = _model[0]
    print()
    print(model.name)

    model.fit(
        train,
        validation_data=test,
        epochs=5,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                SAVE_MODEL_PATH / f"{model.name}.keras",
                save_best_only=True,
            )
        ]
    )


GRU-Att-False
Epoch 1/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - loss: 3.1107 - sparse_categorical_accuracy: 0.5163

2026-10-19 13:08:57.109404: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


1298/1298 ━━━━━━━━━━━━━━━━━━━━ 423s 325ms/step - loss: 3.1107 - sparse_categorical_accuracy: 0.5163 - val_loss: 2.5143 - val_sparse_categorical_accuracy: 0.5679
Epoch 2/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 415s 319ms/step - loss: 2.3462 - sparse_categorical_accuracy: 0.5850 - val_loss: 2.2567 - val_sparse_categorical_accuracy: 0.5898
Epoch 3/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 409s 315ms/step - loss: 2.0999 - sparse_categorical_accuracy: 0.6131 - val_loss: 2.0172 - val_sparse_categorical_accuracy: 0.6297
Epoch 4/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 418s 322ms/step - loss: 1.9022 - sparse_categorical_accuracy: 0.6472 - val_loss: 1.8761 - val_sparse_categorical_accuracy: 0.6551
Epoch 5/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 428s 330ms/step - loss: 1.7552 - sparse_categorical_accuracy: 0.6705 - val_loss: 1.7344 - val_sparse_categorical_accuracy: 0.6781

GRU-Att-True
Epoch 1/5
1298/1298 ━━━━━━━━━━━━━━━━━━━━ 442s 341ms/step - loss: 2.7510 - sparse_categorical_accuracy: 0.5591 - val_loss: 2.1745 - val_sparse

In [ ]:
for _model in models:
    _model[1].save(SAVE_MODEL_PATH / f"{_model[1].name}.keras")
    _model[2].save(SAVE_MODEL_PATH / f"{_model[2].name}.keras")

In [35]:
models

[(<Functional name=GRU-Att-False, built=True>,
  <Functional name=GRU-Att-False_encoder, built=True>,
  <Functional name=GRU-Att-False_decoder, built=True>),
 (<Functional name=GRU-Att-True, built=True>,
  <Functional name=GRU-Att-True_encoder, built=True>,
  <Functional name=GRU-Att-True_decoder, built=True>)]

In [36]:
models = [
    [
        keras.models.load_model(
            SAVE_MODEL_PATH / f"GRU-Att-{att}{part}.keras"
        )
        for part in ('', '_encoder', '_decoder')
    ]
    for att in (False, True)
]
models

[[<Functional name=GRU-Att-False, built=True>,
  <Functional name=GRU-Att-False_encoder, built=True>,
  <Functional name=GRU-Att-False_decoder, built=True>],
 [<Functional name=GRU-Att-True, built=True>,
  <Functional name=GRU-Att-True_encoder, built=True>,
  <Functional name=GRU-Att-True_decoder, built=True>]]

In [37]:
models[0][1].summary()

Model: "GRU-Att-False_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_enc (InputLayer)          │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_enc (Embedding)       │ (None, None, 128)      │     6,432,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_cell_enc (RNN)              │ [(None, None, 128),    │        99,072 │
│                                 │ (None, 128)]           │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,531,968 (24.92 MB)

 Trainable params: 6,531,968 (24.92 MB)

 Non-trainable params: 0 (0.00 B)

# Testing

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

Make a function for text translation. Translate some text and evaluate model performance.

In [81]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    encoder: keras.Model,
    decoder: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text` translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''

    encoded = np.array( [tokenizer.encode(text)] )

    has_attention = any('inf_enc_seq' in inp.name for inp in decoder.inputs)

    enc_seq, *states = encoder.predict(encoded, verbose=0)
    
    current_dec_input = np.array([[tokenizer.bos_token_id]], dtype='int32')
    predicted_ids = []
    
    for _ in range(max_len):
        if has_attention:
            dec_inputs = [current_dec_input, enc_seq] + states
        else:
            dec_inputs = [current_dec_input] + states
        
        preds, *states = decoder.predict(dec_inputs, verbose=0)
        
        next_token = np.argmax(preds[:, -1, :], axis=-1)
        predicted_ids.append(next_token)
        
        if np.all(next_token == tokenizer.eos_token_id):
            break

        current_dec_input = next_token[:, None].astype('int32')
        
    predict_encoded = np.stack(predicted_ids, axis=1)
    
    return tokenizer.decode(predict_encoded)[0]

In [82]:
translate("Я люблю ходить в кино", tokenizer, *models[1][1:])

'I love to go to the movies.<|endoftext|>'

In [90]:
texts = [
    "Я люблю ходить в кино",
    "Меня зовут Вася",
    "Меня зовут и я прихожу",
    "Слушайте, а как мы будем разбираться с этой проблемой",
    "Я хочу кушать",
    "Я",
    "Мы",
    "Дом",
    "Улица",
    "Фонарь",
    "Аптека"
]

In [91]:
for t in texts:
    print(f"RUS:  {t}")
    for i, m in enumerate(models):
        print(f"ENG{i}: {translate(t, tokenizer, *m[1:])}")
    print()

RUS:  Я люблю ходить в кино
ENG0: I like to be a little more careful.<|endoftext|>
ENG1: I love to go to the movies.<|endoftext|>

RUS:  Меня зовут Вася
ENG0: I'm not going to be a little longer<|endoftext|>
ENG1: I'm going to meet you.<|endoftext|>

RUS:  Меня зовут и я прихожу
ENG0: I'm sorry I'm not going to be able to<|endoftext|>
ENG1: I'm going to get rid of my problems.<|endoftext|>

RUS:  Слушайте, а как мы будем разбираться с этой проблемой
ENG0: Please be so careful.<|endoftext|>
ENG1: The radio is the last time we've been working for the<|endoftext|>

RUS:  Я хочу кушать
ENG0: I want to be a little bit.<|endoftext|>
ENG1: I want to travel.<|endoftext|>

RUS:  Я
ENG0: I'm not a doctor.<|endoftext|>
ENG1: <|endoftext|>

RUS:  Мы
ENG0: We're a little bit late.<|endoftext|>
ENG1: <|endoftext|>

RUS:  Дом
ENG0: The children are a beautiful.<|endoftext|>
ENG1: <|endoftext|>

RUS:  Улица
ENG0: The flowers are a good teacher.<|endoftext|>
ENG1: Brazil is a fantastic.<|endoftext|>

R